# Expert-CFG: VQA-RAD Ablation (4 runs)

| Run | γ | LoRA | Attn |
|-----|---|------|------|
| 1_base | 1.0 | N | N |
| 2_cfg | 1.5 | N | N |
| 3_lora | 1.0 | Y | N |
| 4_full | 1.5 | Y | Y |

**필요한 Kaggle 데이터셋:**
```
/kaggle/input/expert-cfg-data/
├── VQA-RAD/images/
└── test.json

/kaggle/input/datasets/ciderchicken/expert-cfg-weights/
└── Phi-3.5-vision-instruct/   # 모델 가중치 (.safetensors 등)
```

## Step 1. 패키지 설치

In [ ]:
!pip install -q \
    "transformers==4.48.3" \
    "accelerate==0.30.0" \
    "peft==0.11.1" \
    bitsandbytes

import importlib
import transformers, tokenizers
importlib.reload(transformers)
print("transformers:", transformers.__version__)   # 4.48.3
print("tokenizers  :", tokenizers.__version__)     # 0.21.x

## Step 2. Expert-CFG 클론

In [ ]:
import os, subprocess, sys, json, shutil

REPO_DIR = '/kaggle/working/Expert-CFG'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone', '--branch', 'main',
    'https://github.com/minseongkim-dev/expert-cfg',
    REPO_DIR,
], check=True)
print('클론 완료')

## Step 3. 모델 가중치 복사

레포 안의 `Phi-3.5-vision-instruct/`에는 커스텀 processor(`processing_phi3_v_cfg.py`)가 있음.  
Kaggle 데이터셋의 가중치(.safetensors 등)를 그 폴더로 복사해서 합침.

In [ ]:
import shutil

WEIGHTS_SRC = '/kaggle/input/datasets/ciderchicken/expert-cfg-weights/Phi-3.5-vision-instruct'
MODEL_DIR   = '/kaggle/working/Expert-CFG/Phi-3.5-vision-instruct'

copied = 0
for fname in os.listdir(WEIGHTS_SRC):
    src = os.path.join(WEIGHTS_SRC, fname)
    dst = os.path.join(MODEL_DIR, fname)
    if os.path.isfile(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'복사: {fname}')
        copied += 1

print(f'완료: {copied}개 파일 복사')

In [ ]:
# 호환성 패치
import json, shutil

# --- HF 모듈 캐시 초기화 ---
modules_cache = os.path.expanduser('~/.cache/huggingface/modules/transformers_modules')
if os.path.exists(modules_cache):
    shutil.rmtree(modules_cache)
    print('✅ transformers_modules 캐시 초기화')

# --- tokenizer_config.json AutoProcessor 교정 ---
# transformers가 tokenizer_config.json을 processor_config.json 이후에 읽어 덮어쓰므로 수정
tok_cfg_path = os.path.join(MODEL_DIR, 'tokenizer_config.json')
if os.path.exists(tok_cfg_path):
    with open(tok_cfg_path) as f:
        tok_cfg = json.load(f)
    old = tok_cfg.get('auto_map', {}).get('AutoProcessor', '')
    if old != 'processing_phi3_v_cfg.Phi3VProcessor':
        tok_cfg.setdefault('auto_map', {})['AutoProcessor'] = 'processing_phi3_v_cfg.Phi3VProcessor'
        with open(tok_cfg_path, 'w') as f:
            json.dump(tok_cfg, f, indent=2)
        print(f'✅ tokenizer_config.json AutoProcessor 교정: {old!r} → processing_phi3_v_cfg.Phi3VProcessor')
    else:
        print('ℹ️  tokenizer_config.json 이미 올바름')
else:
    print('⚠️  tokenizer_config.json 없음 — 건너뜀')

## Step 4. 경로 확인

In [ ]:
DATA_ROOT = '/kaggle/input/expert-cfg-data'
TEST_JSON = f'{DATA_ROOT}/test.json'
IMG_ROOT  = f'{DATA_ROOT}/VQA-RAD/images'
WORK_DIR  = '/kaggle/working'

for label, path in [('model', MODEL_DIR), ('test.json', TEST_JSON), ('images/', IMG_ROOT)]:
    print(f"{('✅' if os.path.exists(path) else '❌')} {label}: {path}")

if os.path.exists(IMG_ROOT):
    print(f'   이미지 수: {len([f for f in os.listdir(IMG_ROOT) if f.endswith(".jpg")])}개')

## Step 5. 4가지 실행

> 각 셀을 순서대로 실행. 셀당 약 1-2시간.

In [ ]:
# Run 1: baseline (γ=1.0, no LoRA, no Attn)
cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--save_path',          f'{WORK_DIR}/result_1_base.json',
]
print('▶ Run 1: base')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 2: CFG only (γ=1.5, no LoRA, no Attn)
cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--cfg',                '1.5',
    '--bf16',
    '--num_crops',          '4',
    '--save_path',          f'{WORK_DIR}/result_2_cfg.json',
]
print('▶ Run 2: cfg')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 3: LoRA only (γ=1.0, LoRA, no Attn)
cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--bf16',
    '--num_crops',          '4',
    '--save_path',          f'{WORK_DIR}/result_3_lora.json',
]
print('▶ Run 3: lora')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 4: Expert-CFG full (γ=1.5, LoRA, Attn)
cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--cfg',                '1.5',
    '--attn',               '3.0',
    '--use_attn_patch',
    '--use_qlora',
    '--bf16',
    '--num_crops',          '4',
    '--save_path',          f'{WORK_DIR}/result_4_full.json',
]
print('▶ Run 4: Expert-CFG full')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

## Step 6. 결과 비교

In [ ]:
runs = [
    ('1_base', f'{WORK_DIR}/result_1_base.json', '1.0', 'N', 'N'),
    ('2_cfg',  f'{WORK_DIR}/result_2_cfg.json',  '1.5', 'N', 'N'),
    ('3_lora', f'{WORK_DIR}/result_3_lora.json', '1.0', 'Y', 'N'),
    ('4_full', f'{WORK_DIR}/result_4_full.json', '1.5', 'Y', 'Y'),
]

print(f"{'Run':<30} {'Closed':>8} {'Open':>8} {'Overall':>9}  γ  LoRA  Attn")
print('-' * 75)

for name, path, cfg, lora, attn in runs:
    if not os.path.exists(path):
        print(f"{'vqa-rad_' + name:<30} {'(미실행)':>8}")
        continue
    with open(path) as f:
        out = json.load(f)
    s = out['summary']
    closed_acc = s.get('closed_accuracy', 0) * 100
    open_acc   = s.get('open_accuracy', 0) * 100
    overall    = s['accuracy'] * 100
    print(f"{'vqa-rad_' + name:<30} {closed_acc:>7.1f}%  {open_acc:>6.1f}%  {overall:>7.1f}%  {cfg}   {lora}     {attn}")